# Paper tables (main text and supplementary material)

This notebook builds every table of the paper from the CSV files written by
`pairwise_analysis`, `clustering_analysis`, `seed_clustering_analysis`,
`elsst_diagnostic_analysis` and `apa_diagnostic_analysis`. It computes nothing new.

Each table is saved twice in `analysis/results/paper/`: as CSV and as a LaTeX table (`.tex`, booktabs).
Main-text tables start with `table`, supplementary tables with `S`.

Run it after both diagnostic notebooks.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "analysis").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display, Markdown

from scripts import config

OUT_DIR = config.PROJECT_ROOT / "analysis/results/paper"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAMES = {"elsst": "ELSST", "apa": "APA"}
TECHNIQUE_ORDER = ["Clustering", "Pairwise", "Seeded"]
MODEL_ORDER = [config.MODELS[m]["display_name"] for m in config.MODELS] + [config.BASELINE_DISPLAY_NAME]

# Model keys (Hugging Face ids) to display names, for the calibration files
KEY_TO_NAME = {key: info["display_name"] for key, info in config.MODELS.items()}
KEY_TO_NAME[config.BASELINE_KEY] = config.BASELINE_DISPLAY_NAME

# Both diagnostic notebooks must have been run with the current code
# (seeded_draws.csv only exists in the current version of the results).
for key, name in DATASET_NAMES.items():
    if not (config.RESULTS_DIRS[key] / "seeded_draws.csv").exists():
        raise FileNotFoundError(f"The {name} results are missing or from an older version. "
                                f"Run {key}_diagnostic_analysis.ipynb first.")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Tables are written to", OUT_DIR)

In [ ]:
# Helpers: read the result files of both data sets, order rows, save CSV and LaTeX.

def read_results(file_name):
    """One CSV from each data set, stacked, with a 'dataset' column (ELSST first)."""
    frames = []
    for key, name in DATASET_NAMES.items():
        path = config.RESULTS_DIRS[key] / file_name
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing. Run the {name} diagnostic notebook first.")
        df = pd.read_csv(path)
        df.insert(0, "dataset", name)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def tidy(df):
    """Capitalised technique names, rows in the order data set > technique > model."""
    df = df.copy()
    if "technique" in df:
        df["technique"] = df["technique"].str.capitalize()
        df["technique"] = pd.Categorical(df["technique"], TECHNIQUE_ORDER, ordered=True)
    if "model" in df:
        df["model"] = pd.Categorical(df["model"], MODEL_ORDER, ordered=True)
    if "dataset" in df:
        df["dataset"] = pd.Categorical(df["dataset"], list(DATASET_NAMES.values()), ordered=True)
    sort_cols = [col for col in ["dataset", "technique", "model"] if col in df]
    df = df.sort_values(sort_cols).reset_index(drop=True)
    for col in ["dataset", "technique", "model"]:
        if col in df:
            df[col] = df[col].astype(str)
    return df


def latex_escape(text):
    for old, new in [("\\", "\\textbackslash{}"), ("&", "\\&"), ("%", "\\%"), ("_", "\\_"), ("#", "\\#")]:
        text = text.replace(old, new)
    return text


LABEL_COLUMNS = {"Dataset", "Technique", "Model", "Split", "Pipeline A", "Pipeline B",
                 "Selected parameters", "Overlap band", "Variation"}


def to_latex(df, caption, label):
    """A simple booktabs table. Numbers are written as they appear in df (format them first)."""
    align = "".join("l" if col in LABEL_COLUMNS else "r" for col in df.columns)
    lines = ["\\begin{table}[ht]", "\\centering", "\\small",
             f"\\caption{{{latex_escape(caption)}}}", f"\\label{{{label}}}",
             f"\\begin{{tabular}}{{{align}}}", "\\toprule",
             " & ".join(latex_escape(str(col)) for col in df.columns) + " \\\\", "\\midrule"]
    for _, row in df.iterrows():
        lines.append(" & ".join(latex_escape(str(v)) for v in row.values) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines) + "\n"


def save_table(df, name, caption):
    """Save as CSV and LaTeX, and show it."""
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    (OUT_DIR / f"{name}.tex").write_text(to_latex(df, caption, f"tab:{name}"))
    display(Markdown(f"**{name}.** {caption}"))
    display(df)


def fmt(df, decimals=3, columns=None):
    """Round numeric columns to strings with a fixed number of decimals (empty for missing values)."""
    df = df.copy()
    for col in (columns or df.select_dtypes("number").columns):
        df[col] = df[col].map(lambda v: "" if pd.isna(v) else f"{v:.{decimals}f}")
    return df

---
## 1. Main-text tables

### Table 1. Primary performance

In [ ]:
perf = tidy(read_results("primary_performance.csv"))
ci = tidy(read_results("bootstrap_intervals.csv"))
ci = ci[ci["metric"] == "f1"][["dataset", "technique", "model", "ci_low", "ci_high"]]
perf = perf.merge(ci, on=["dataset", "technique", "model"], how="left")

table1 = pd.DataFrame({
    "Dataset": perf["dataset"],
    "Technique": perf["technique"],
    "Model": perf["model"],
    "Precision": perf["precision"].map("{:.3f}".format),
    "Recall": perf["recall"].map("{:.3f}".format),
    "F1 [95% CI]": [f"{f:.3f} [{lo:.3f}, {hi:.3f}]" for f, lo, hi in zip(perf["f1"], perf["ci_low"], perf["ci_high"])],
    "FP per 10,000": perf["fp_per_10k_negatives"].map("{:.2f}".format),
    "F1 at ELSST prevalence": perf["f1_at_reference_prevalence"].map("{:.3f}".format),
})
save_table(table1, "table1_primary_performance",
           "Precision, recall and F1 by data set, technique and model. 95% intervals from 1,000 concept-level "
           "bootstrap samples (seeded clustering: mean over 10 seed draws, interval from the median draw). "
           "FP per 10,000: false positives per 10,000 non-synonym pairs. F1 at ELSST prevalence: precision "
           "recomputed with the class balance of the ELSST test split, recall and false-positive rate unchanged.")

### Table 2. Lexical trap test

In [ ]:
trap = tidy(read_results("lexical_trap.csv"))
table2 = fmt(pd.DataFrame({
    "Dataset": trap["dataset"], "Technique": trap["technique"], "Model": trap["model"],
    "FPR (hard)": trap["hard_neg_fpr"], "FPR (easy)": trap["easy_neg_fpr"],
}), 4)
table2["Delta FPR"] = trap["delta_fpr"].map("{:+.3f}".format)
n_hard = trap.groupby("dataset")["n_hard"].first()
save_table(table2, "table2_lexical_trap",
           "Lexical trap test. False-positive rate on hard negatives (character 3-gram Jaccard > 0.5) and easy "
           f"negatives (Jaccard = 0). N hard = {n_hard['ELSST']:,} on ELSST and {n_hard['APA']:,} on APA. "
           "The 3-gram baseline flags every hard negative by construction and is shown for reference.")

### Table 3. Semantic decay test

In [ ]:
decay = tidy(read_results("semantic_decay.csv"))
table3 = fmt(pd.DataFrame({
    "Dataset": decay["dataset"], "Technique": decay["technique"], "Model": decay["model"],
    "Recall (easy)": decay["recall_easy"], "Recall (hard)": decay["recall_hard"],
    "Retention": decay["retention_rate"],
}))
table3.insert(5, "Delta recall", decay["delta_recall"].map("{:+.3f}".format))
n_easy = decay.groupby("dataset")["n_easy"].first()
n_hard = decay.groupby("dataset")["n_hard"].first()
save_table(table3, "table3_semantic_decay",
           "Semantic decay test. Recall on easy positives (Jaccard > 0.5) and hard positives (Jaccard = 0). "
           "Delta recall = recall (hard) - recall (easy); retention = recall (hard) / recall (easy). "
           f"N easy / hard = {n_easy['ELSST']:,} / {n_hard['ELSST']:,} on ELSST and "
           f"{n_easy['APA']:,} / {n_hard['APA']:,} on APA.")

### Table 4. Embedding stability test

In [ ]:
stab = tidy(read_results("stability.csv"))
base = stab[stab["noise_level"] == 0.0].set_index(["dataset", "technique", "model"])
top = stab["noise_level"].max()
high = stab[stab["noise_level"] == top].set_index(["dataset", "technique", "model"])
seed_sd = base["sd_f1"].where(base.index.get_level_values("technique") == "Seeded")

table4 = pd.DataFrame({
    "F1 (sigma = 0)": base["mean_f1"],
    "SD over seed draws": seed_sd,
    f"F1 (sigma = {top:.2f})": high["mean_f1"],
    f"SD (sigma = {top:.2f})": high["sd_f1"],
    f"Delta F1 (sigma = {top:.2f})": high["mean_delta_f1"],
    f"Agreement (sigma = {top:.2f})": high["mean_agreement"],
}).reset_index()
table4 = tidy(table4).rename(columns={"dataset": "Dataset", "technique": "Technique", "model": "Model"})
table4 = fmt(table4)
save_table(table4, "table4_embedding_stability",
           f"Embedding stability test. Gaussian noise added to the embeddings, {config.N_RELIABILITY_RUNS} runs per "
           "noise level. Delta F1: mean change from the noise-free run. Agreement: Jaccard overlap between the "
           "predicted synonym pairs of a noisy run and of the noise-free run. Seeded clustering at sigma = 0 varies "
           f"only the seed draw ({config.N_SEED_DRAWS} draws); at sigma > 0 the seed draw is fixed. "
           "All noise levels are in Supplementary Table S3.")

### Table 5. Rare-word test

In [ ]:
rare = tidy(read_results("rare_word.csv"))
bands = tidy(read_results("rare_word_by_overlap.csv"))
pooled = bands[bands["overlap_band"] == "pooled"][["dataset", "technique", "model", "delta_recall"]]
rare = rare.merge(pooled.rename(columns={"delta_recall": "delta_within"}), on=["dataset", "technique", "model"])

table5 = fmt(pd.DataFrame({
    "Dataset": rare["dataset"], "Technique": rare["technique"], "Model": rare["model"],
    "Recall (common)": rare["recall_common"], "Recall (rare)": rare["recall_rare"],
}))
table5["Delta recall"] = rare["delta_recall"].map("{:+.3f}".format)
table5["Delta recall within overlap bands"] = rare["delta_within"].map("{:+.3f}".format)
n_rare = rare.groupby("dataset")["n_rare"].first()
save_table(table5, "table5_rare_word",
           "Rare-word test. Recall on positive pairs in the top (common) and bottom (rare) decile of word frequency. "
           "Delta recall = recall (common) - recall (rare). The last column compares rare and common pairs within "
           "the same lexical-overlap band (Jaccard = 0, 0 to 0.5, > 0.5) and pools the bands; values near zero mean "
           "that the raw gap is explained by lexical overlap. "
           f"N rare = {n_rare['ELSST']:,} on ELSST and {n_rare['APA']:,} on APA. Bands: Supplementary Table S4.")

### Table 6. Structural alignment test

In [ ]:
# Part A: one value per model and data set
rho = tidy(read_results("structural_alignment.csv"))
table6a = rho.pivot(index="model", columns="dataset", values="spearman_rho")
table6a = table6a.reindex([m for m in MODEL_ORDER if m in table6a.index])[list(DATASET_NAMES.values())]
n_pairs = rho.groupby("dataset")["n_pairs"].first()
table6a = fmt(table6a.reset_index().rename(columns={"model": "Model", "ELSST": "rho ELSST", "APA": "rho APA"}))
save_table(table6a, "table6a_structural_similarity",
           "Structural alignment, part A. Spearman rho between similarity (cosine; Jaccard for the baseline) and "
           "thesaurus path length among non-synonym pairs with a path (length >= 2). A negative rho means that "
           f"similarity falls as concepts are further apart. Pairs: {n_pairs['ELSST']:,} on ELSST and "
           f"{n_pairs['APA']:,} on APA (rho on a fixed random sample of 5 million pairs).")

In [ ]:
# Part B: false-merge rate by path length, per pipeline
merge = tidy(read_results("structural_false_merge_by_distance.csv"))
band_cols = [col for col in merge.columns if col.startswith("fpr_")]
table6b = merge[["dataset", "technique", "model"] + band_cols + ["near_miss_share"]].copy()
table6b = fmt(table6b, 4, band_cols)
table6b = fmt(table6b, 3, ["near_miss_share"])
table6b.columns = (["Dataset", "Technique", "Model"]
                   + ["Path " + col.replace("fpr_path_", "").replace("fpr_not_connected", "none") for col in band_cols]
                   + ["Near-miss share"])
save_table(table6b, "table6b_structural_false_merges",
           "Structural alignment, part B. False-positive rate of each pipeline by thesaurus path length between the "
           "two concepts (none: not connected). Near-miss share: fraction of all false positives at path length 2.")

---
## 2. Supplementary tables

### S1. Pair counts

In [ ]:
def n_rows(folder, file_name):
    path = folder / file_name
    return pq.ParquetFile(path).metadata.num_rows if path.exists() else np.nan

rows = []
for key, label in [("elsst_train", "ELSST train (calibration)"), ("elsst_test", "ELSST test"), ("apa", "APA (full)")]:
    folder, prefix = config.DATASETS[key]["folder"], config.DATASETS[key]["prefix"]
    rows.append({
        "Split": label,
        "Concepts": n_rows(folder, f"{prefix}concepts.parquet"),
        "Terms": n_rows(folder, f"{prefix}terms.parquet"),
        "Synonym pairs": n_rows(folder, f"{prefix}positive_pairs.parquet"),
        "Non-synonym pairs": n_rows(folder, f"{prefix}negative_pairs.parquet"),
        "Broader/narrower pairs": n_rows(folder, f"{prefix}hierarchy_pairs.parquet"),
    })
s1 = pd.DataFrame(rows)
for col in s1.columns[1:]:
    s1[col] = s1[col].map(lambda v: "" if pd.isna(v) else f"{int(v):,}")
save_table(s1, "S1_pair_counts",
           "Pair counts. Non-synonym pairs combine every label of one concept with every label of another concept, "
           "excluding concepts on the same family line (all ancestors and descendants). Broader/narrower pairs are "
           "kept apart from the benchmark (Supplementary Table S5).")

### S2. Calibration on the ELSST training split

In [ ]:
cal_dir = config.CALIBRATION_DIR
rows = []

pw = pd.read_csv(cal_dir / "pairwise_results.csv")
for key, sub in pw.groupby("model"):
    best = sub.loc[sub["f1_mean"].idxmax()]
    rows.append({"Technique": "Pairwise", "Model": KEY_TO_NAME.get(key, key),
                 "Selected parameters": f"threshold = {best['threshold']:.2f}",
                 "Precision": best["precision_mean"], "Recall": best["recall_mean"], "F1": best["f1_mean"]})

cl = pd.read_csv(cal_dir / "clustering_results.csv")
for name, sub in cl.groupby("model"):
    best = sub.loc[sub["f1"].idxmax()]
    dims = "none" if best["n_components"] >= 768 else int(best["n_components"])
    rows.append({"Technique": "Clustering", "Model": KEY_TO_NAME.get(name, name),
                 "Selected parameters": f"UMAP dims = {dims}, min cluster size = {int(best['min_cluster_size'])}, "
                                        f"min samples = {int(best['min_samples'])}",
                 "Precision": best["precision"], "Recall": best["recall"], "F1": best["f1"]})

sd = pd.read_csv(cal_dir / "seed_cluster_results.csv")
sd_mean = sd.groupby(["model_name", "num_initial_seeds", "threshold"])[["precision", "recall", "f1"]].mean().reset_index()
for key, sub in sd_mean.groupby("model_name"):
    best = sub.loc[sub["f1"].idxmax()]
    rows.append({"Technique": "Seeded", "Model": KEY_TO_NAME.get(key, key),
                 "Selected parameters": f"seeds = {int(best['num_initial_seeds'])}, threshold = {best['threshold']:.2f}",
                 "Precision": best["precision"], "Recall": best["recall"], "F1": best["f1"]})

s2 = tidy(pd.DataFrame(rows).rename(columns={"Technique": "technique", "Model": "model"}))
s2 = fmt(s2.rename(columns={"technique": "Technique", "model": "Model"}))
save_table(s2, "S2_calibration",
           "Parameters selected by grid search on the ELSST training split (highest F1; seeded clustering: mean "
           "over trials) and the training-split scores. These values are fixed in scripts/config.py for all "
           "evaluations.")

### S3. Embedding stability at every noise level

In [ ]:
s3 = stab[["dataset", "technique", "model", "noise_level", "variation_source", "n_runs",
           "mean_f1", "sd_f1", "mean_delta_f1", "mean_agreement", "min_agreement"]].copy()
s3 = fmt(s3, 3, ["mean_f1", "sd_f1", "mean_delta_f1", "mean_agreement", "min_agreement"])
s3.columns = ["Dataset", "Technique", "Model", "Sigma", "Variation", "Runs",
              "Mean F1", "SD F1", "Delta F1", "Mean agreement", "Min agreement"]
save_table(s3, "S3_stability_all_levels",
           "Embedding stability at every noise level. Agreement: Jaccard overlap of predicted synonym pairs with the "
           "noise-free run.")

### S4. Rare-word gap by lexical-overlap band

In [ ]:
s4 = bands[bands["overlap_band"] != "pooled"].copy()
s4 = fmt(s4[["dataset", "technique", "model", "overlap_band", "n_rare", "n_common", "recall_rare", "recall_common"]],
         3, ["recall_rare", "recall_common"])
s4["delta"] = bands[bands["overlap_band"] != "pooled"]["delta_recall"].map("{:+.3f}".format).values
s4.columns = ["Dataset", "Technique", "Model", "Overlap band", "N rare", "N common",
              "Recall (rare)", "Recall (common)", "Delta recall"]
save_table(s4, "S4_rare_word_by_overlap",
           "Rare-word gap within lexical-overlap bands (character 3-gram Jaccard of the two terms). "
           "Delta recall = recall (common) - recall (rare).")

### S5. Broader/narrower pairs

In [ ]:
hier = tidy(read_results("hierarchy_merge_rate.csv"))
s5 = fmt(hier[["dataset", "technique", "model", "merge_rate_depth_1", "merge_rate_depth_2",
               "merge_rate_depth_3+", "merge_rate_depth_all", "fpr_benchmark_negatives"]], 4,
         ["merge_rate_depth_1", "merge_rate_depth_2", "merge_rate_depth_3+", "merge_rate_depth_all",
          "fpr_benchmark_negatives"])
s5.columns = ["Dataset", "Technique", "Model", "Depth 1", "Depth 2", "Depth 3+", "All depths",
              "FPR benchmark negatives"]
counts = hier.groupby("dataset")[["n_depth_1", "n_depth_2", "n_depth_3+"]].first()
save_table(s5, "S5_broader_narrower",
           "Share of broader/narrower pairs predicted as synonyms, by number of broader steps between the two "
           "concepts (depth 1 = direct parent). These pairs are not part of the benchmark. Pairs at depth 1 / 2 / 3+: "
           + "; ".join(f"{ds} {int(r['n_depth_1']):,} / {int(r['n_depth_2']):,} / {int(r['n_depth_3+']):,}"
                       for ds, r in counts.iterrows()) + ".")

### S6. Paired F1 differences

In [ ]:
diffs = read_results("bootstrap_f1_differences.csv")
s6 = fmt(diffs[["dataset", "pipeline_a", "pipeline_b", "f1_difference", "ci_low", "ci_high",
                "share_samples_a_better"]])
s6.columns = ["Dataset", "Pipeline A", "Pipeline B", "F1 difference (A - B)", "CI low", "CI high",
              "Share of samples A better"]
save_table(s6, "S6_f1_differences",
           "Paired F1 differences between pipelines with 95% concept-level bootstrap intervals (same 1,000 samples "
           "for every pipeline). An interval that excludes 0 marks a difference beyond concept sampling variation.")

### S7. Ranking: top-k recall and correct matches among reviewed candidates

In [ ]:
rank = tidy(read_results("topk_and_review.csv"))
k_cols = [f"recall_at_{k}" for k in config.TOPK_LIST]
n_cols = [f"correct_in_top_{n}" for n in config.REVIEW_BUDGETS]
s7 = fmt(rank[["dataset", "model"] + k_cols + n_cols], 3, k_cols)
s7.columns = (["Dataset", "Model"] + [f"Recall@{k}" for k in config.TOPK_LIST]
              + [f"Correct in top {n:,}" for n in config.REVIEW_BUDGETS])
save_table(s7, "S7_ranking",
           "Recall@k: share of synonym pairs (both directions) whose synonym is among the k nearest terms "
           "(ties: expected value under random order). Correct in top N: synonym pairs among the N most "
           "similar benchmark pairs, as a curator reviewing a ranked list would see them.")

### S8. F1 with preferred-label negatives only

In [ ]:
s8 = fmt(perf[["dataset", "technique", "model", "f1", "f1_pref_pref_only"]])
s8.columns = ["Dataset", "Technique", "Model", "F1 (all-label negatives)", "F1 (preferred-label negatives only)"]
save_table(s8, "S8_preferred_label_negatives",
           "F1 with all-label non-synonym pairs (main analysis) and with non-synonym pairs between preferred labels "
           "only, as in the submitted version.")

### S9. Seeded clustering over seed draws

In [ ]:
draws = read_results("seeded_draws.csv")
draws["model"] = draws["model"].map(lambda m: KEY_TO_NAME.get(m, m))
s9 = draws.groupby(["dataset", "model"])["f1"].agg(["mean", "std", "min", "max"]).reset_index()
s9 = fmt(tidy(s9))
s9.columns = ["Dataset", "Model", "Mean F1", "SD F1", "Min F1", "Max F1"]
save_table(s9, "S9_seed_draws",
           f"Seeded clustering over {config.N_SEED_DRAWS} random seed draws (seeds sampled uniformly from all terms, "
           "no gold information).")

### S10. Mean similarity by path length

In [ ]:
sim = tidy(read_results("structural_similarity_by_distance.csv"))
s10 = sim.pivot_table(index=["dataset", "model"], columns="path_length", values="mean_similarity", observed=True)
s10 = fmt(tidy(s10.reset_index()))
s10 = s10.rename(columns={"dataset": "Dataset", "model": "Model"})
save_table(s10, "S10_similarity_by_path_length",
           "Mean similarity of non-synonym pairs by thesaurus path length (cosine; Jaccard for the baseline).")

---
## 3. Files written

In [ ]:
for path in sorted(OUT_DIR.glob("*.csv")):
    print(path.name)